# Notebook 10: Línia d'atenció contra la violència masclista 900 900 120

Enlace a la fuente: https://analisi.transparenciacatalunya.cat/Societat-benestar/L-nia-d-atenci-contra-la-viol-ncia-masclista-900-9/q2sg-894k/about_data

## Visualizaciones objetivo

- Catalunya:

    - [dxvxm-catalunya-v_5_1-cat](https://app.flourish.studio/visualisation/5760894/edit): impacto COVID (vis. permanente); line chart de llamadas por mes, una línea para 2019 y otra para 2020

- Santa Coloma de Gramenet:

    - Básicamente son comparaciones entre Catalunya y el Barcelonés, excepto una que es genérica para Cataluña, pero no está en DxVxD.

    - El año siempre está en la X.

    - Cuestión importante a resolver: ¿se distingue entre mujeres que llaman vs llamadas, como se hace en el del SIAD?

    - Stacked bar chart relativos:

        - Selección entre Cat y Barcelonés:

            - [dxvxm-santa_coloma_g-v_1_1_1-esp](https://app.flourish.studio/visualisation/13535838/edit), ([-cat](https://app.flourish.studio/visualisation/13252855/edit)): % de llamadas hechas por la propia persona, familiar, profesional u otros

            - [dxvxm-santa_coloma_g-v_1_1_2-esp](https://app.flourish.studio/visualisation/29806702/edit), ([-cat](https://app.flourish.studio/visualisation/13252880/edit)): % de llamadas según el vínculo entre la persona que llama y la víctima (¿solamente para cuando llama un familiar?)

        - Grouped (no hace falta seleccionar porque están uno al lado del otro):

            - [dxvxm-santa_coloma_g-v_1_1_3_1-esp](https://app.flourish.studio/visualisation/13535962/edit), ([-cat](https://app.flourish.studio/visualisation/13252887/edit)): % de ¿llamadas o mujeres que llaman? según la edad (¿de la víctima o de la persona que llama?)

                - Caída en picado de la calidad a partir de 2023

            - [dxvxm-santa_coloma_g-v_1_1_3_2-esp](https://app.flourish.studio/visualisation/13535958/edit), ([-cat](https://app.flourish.studio/visualisation/29807116/edit)): % de ¿llamadas o mujeres que llaman? según nivel de estudios
            
                - Esta estaría bien compararla con el % de la población que tiene esos estudios, de lo contrario no sabemos si hay diferencia o no (y es solo que hay menos gente con estudios)

                - La calidad de los datos ha caído en picado y el 2025 el 98% es "no consta"

    - Heat maps absolutos del número de llamadas por año (X) y una variable (Y), que permiten seleccionar Cat o Barcelonés

        - Caída en picado de la calidad a partir de 2023

        - [dxvxm-santa_coloma_g-v_1_3_1-esp](https://app.flourish.studio/visualisation/13536280/edit), ([-cat](https://app.flourish.studio/visualisation/13252934/edit)): grupos de tiempo de relación en Y

        - [dxvxm-santa_coloma_g-v_1_3_2-esp](https://app.flourish.studio/visualisation/29841707/edit), ([-cat](https://app.flourish.studio/visualisation/13252938/edit)): grupos de edad en Y

    - Line chart:

        - [dxvxm-santa_coloma_g-v_2_1_1-esp](https://app.flourish.studio/visualisation/13536525/edit), ([-cat](https://app.flourish.studio/visualisation/13253632/edit)): line chart de las llamadas (Y) por año (X) en Cataluña


In [1]:
import pandas as pd
import requests

from pathlib import Path

from utils import *

PATH_DATA = Path("./data")

In [ ]:
# TODO: instrucciones de descarga en el Sheets

## Preámbulo

De este dataset nos interesa:

- Desagregar por:
    
    - Provincias ("Territori")

    - "Any"

    - Otras variables:
        - "La pròpia usuària" vs "Familiar o persona propera" vs "Professional" vs "Altres"
        - "Progenitors o progenitores" vs "Amistats" vs "Germans o germanes" vs "Fills o filles" vs "Altres" vs "No consta"
        - Grupos de edad ("Edat)
        - Nivel de estudios
        - Grupos de tiempo de relación ("Temps")

Cuestión pendiente: "Ús previ servei" puede servir para distinguir llamadas de mujeres. ¿Se debe hacer?

Generación del enlace:

1. En el "Gestor de columna", quitar todas las columnas excepto **Any, Comarca, Sexe, Edat, Persona que truca, Detall familiar o persona propera, Formació, Motiu Trucada, Relació agressor, Temps de relació**. Aplicar cambios.

    - En este punto, el dataset tiene unas 65.000 filas.

2. En "Agrupar i agregar", en "Agrega per", seleccionar el cálculo "compte de files". Luego, en "Agrupar per", poner todas las columnas seleccionadas. Aplicar cambios.

    - Esto baja de 65.000 filas a 13.000.

3. Ahora habría que exportar, pero como son más de 1000 filas, no se puede generar el enlace. Igualmente, copiamos en enlace SODA 2 formato JSON.

    - El ChatGPT propone añadir `&$limit=1000` al final del enlace, pero esto no se ha conseguido que funcionase.

    - En su lugar, podemos generar una consulta usando requests. Hay dos maneras:
     
        - Una forma es más manual: seguimos los pasos anteriormente descritos y le pedimos al ChatGPT, pasándole en enlace, que "genere la misma consulta pero usando requests con $query".

In [ ]:
# Este es el URL que genera la web
    # Se lo podemos pasar al Chat para que lo "traduza" a SQL
url_web = "https://analisi.transparenciacatalunya.cat/resource/q2sg-894k.json?$query=SELECT%0A%20%20count(*)%2C%0A%20%20%60any%60%2C%0A%20%20%60comarca%60%2C%0A%20%20%60sexe%60%2C%0A%20%20%60edat%60%2C%0A%20%20%60personaquetruca%60%2C%0A%20%20%60detallfamiliarpersonaquetruca%60%2C%0A%20%20%60formacio%60%2C%0A%20%20%60motiutrucada%60%2C%0A%20%20%60relacioagressor%60%2C%0A%20%20%60tempsderelacio%60%0AGROUP%20BY%0A%20%20%60any%60%2C%0A%20%20%60comarca%60%2C%0A%20%20%60sexe%60%2C%0A%20%20%60edat%60%2C%0A%20%20%60personaquetruca%60%2C%0A%20%20%60detallfamiliarpersonaquetruca%60%2C%0A%20%20%60formacio%60%2C%0A%20%20%60motiutrucada%60%2C%0A%20%20%60relacioagressor%60%2C%0A%20%20%60tempsderelacio%60"

In [15]:
# Código único de la tabla
resource_code = "q2sg-894k"

### Metadatos

In [ ]:
url_metadata = f"https://analisi.transparenciacatalunya.cat/api/views/{resource_code}.json"

metadata = requests.get(url_metadata).json()

df_columns = pd.DataFrame([
    {
        "fieldName": column.get("fieldName"),
        "name": column.get("name"),
        "dataTypeName": column.get("dataTypeName")
    }
    for column in metadata["columns"]
])

print(list(df_columns["fieldName"]))

display(df_columns)

['data', 'any', 'duradasegons', 'diasetmana', 'mes', 'hora', 'llengua', 'comarca', 'municipi', 'ambitterritorial', 'provincia', 'sexe', 'edat', 'estatcivil', 'personaquetruca', 'detallfamiliarpersonaquetruca', 'detallprofessional', 'situaciofamiliar', 'situaciofillsfilles', 'nombrefillsfilles', 'situaciofillsfilles_a', 'nombrefillsfilles_a', 'pateixenmaltractament', 'nacionalitatagrupada', 'situaciolegal', 'tempsresidencia', 'formacio', 'situaciolaboral', 'detallsituaciolaboral', 'uspreviservei', 'motiutrucada', 'tipusassessorament', 'ambitviolenciamasclista', 'detallambit', 'v_fisica', 'v_psicologica', 'v_sexual', 'v_economica', 'conviuagressor', 'relacioagressor', 'relacioagressorambitlaboral', 'tempsderelacio', 'nacionalitatagressor', 'derivacio', 'xarxa', 'derivacio_a', 'xarxa_a', 'derivacio_b', 'xarxa_b', 'derivacio_c', 'xarxa_c', 'traduccio', 'idiomatraduccio', 'reclamacionsqueixes', 'emergencia', 'connexi_', 'motiu_trucada_2', '_mbit_viol_ncia_2', 'tipus_assessorament_2', 'detal

,fieldName,name,dataTypeName
0,data,Data,calendar_date
1,any,Any,number
2,duradasegons,Durada segons,text
3,diasetmana,Dia setmana,text
4,mes,Mes,text
...,...,...,...
65,viol_ncia_digital,Violència digital,text
66,viol_ncia_obst_trica,Violència Obstètrica,text
67,viol_ncia_de_segon_ordre,Violència de segon ordre,text
68,viol_ncia_vic_ria,Violència Vicària,text


In [ ]:
url_data = f"https://analisi.transparenciacatalunya.cat/resource/{resource_code}.json"

## Descarga por API, en subgrupos

### dxvxm-santa_coloma_g-v_2_1_1

In [51]:
query = """
SELECT
    `any`,
    count(*) AS Trucades
WHERE
    `sexe`="Dona" AND `motiutrucada`="Violència masclista" AND `provincia` IN ("Barcelona", "Girona", "Lleida", "Tarragona")
GROUP BY
    `any`
ORDER BY
    `any`
"""

response = requests.get(
    url_data,
    params={"$query": query},
    timeout=60
)
check_raw_dades_obertes(response)

df_atenciones_por_anyo = pd.DataFrame(response.json())

print(df_atenciones_por_anyo.info())
df_atenciones_por_anyo.head(20)

<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   any       5 non-null      str  
 1   Trucades  5 non-null      str  
dtypes: str(2)
memory usage: 254.0 bytes
None


,any,Trucades
0,2022,6156
1,2023,10989
2,2024,10033
3,2025,9699
4,2026,9035


In [ ]:
# TODO: LOS DATOS ANTERIORES DATOS ESTÁN MAL
    # Nota en el Obsidian: 2026-11-10

In [35]:
# Datos del Google Drive para esta visualización
df_atenciones_tel_drive = pd.read_csv(PATH_DATA / "CSV_DataSetComplert 2026 Línia atencio contra la violencia masclista 900 900 120.csv", sep=";")
df_atenciones_tel_drive.head()

/tmp/ipykernel_575103/3584365549.py:2: DtypeWarning: Columns (15: Detall familiar o persona proopera, 56: Motiu trucada 2, 57: Àmbit violència 2, 58: Tipus assessorament 2, 59: Detall àmbit de la violència, 60: Motiu trucada 3, 61: Àmbit de la violència 3, 62: Tipus assessorament 3, 63: Detall àmbit 3, 64: Violència psicològica ambiental, 65: Violència digital, 66: Violència Obstètrica, 67: Violència de segon ordre, 68: Violència Vicària, 69: Violència detectada, 70: Detall familiar o persona propera) have mixed types. Specify dtype option on import or set low_memory=False.
  df_atenciones_tel_drive = pd.read_csv(PATH_DATA / "CSV_DataSetComplert 2026 Línia atencio contra la violencia masclista 900 900 120.csv", sep=";")


,Data,Any,Durada segons,Dia setmana,Mes,Hora,Llengua,Comarca,Àmbit territorial,Provincia,...,Àmbit de la violència 3,Tipus assessorament 3,Detall àmbit 3,Violència psicològica ambiental,Violència digital,Violència Obstètrica,Violència de segon ordre,Violència Vicària,Violència detectada,Detall familiar o persona propera
0,01/01/2013,2013,Entre 5 i 15 minuts,Dimarts,Gener,00/01/1900 1:00,Castellà,Barcelonès,Metropolità,Barcelona,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,01/01/2013,2013,Menys de 5 minuts,Dimarts,Gener,00/01/1900 3:20,Castellà,Barcelonès,Metropolità,Barcelona,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,01/01/2013,2013,Entre 5 i 15 minuts,Dimarts,Gener,00/01/1900 19:07,Castellà,Barcelonès,Metropolità,Barcelona,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,01/01/2013,2013,Menys de 5 minuts,Dimarts,Gener,00/01/1900 16:37,Castellà,Barcelonès,Metropolità,Barcelona,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,01/01/2013,2013,Entre 5 i 15 minuts,Dimarts,Gener,00/01/1900 18:02,Castellà,Barcelonès,Metropolità,Barcelona,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
df_atenciones_tel_drive_1 = df_atenciones_tel_drive[
    (df_atenciones_tel_drive["Sexe"] == "Dona") &
    #(df_atenciones_tel_drive["Ús previ servei"] == "No") &     # Filtro que no encaja
    (df_atenciones_tel_drive["Motiu Trucada"] == "Violència masclista") &
    (df_atenciones_tel_drive["Provincia"].isin(["Barcelona", "Girona", "Lleida", "Tarragona"]))  # Hay muchas llamadas desde otras provincias de España o llamadas en las que la provincia no consta

]
df_atenciones_tel_drive_1.head(3)

,Data,Any,Durada segons,Dia setmana,Mes,Hora,Llengua,Comarca,Àmbit territorial,Provincia,...,Àmbit de la violència 3,Tipus assessorament 3,Detall àmbit 3,Violència psicològica ambiental,Violència digital,Violència Obstètrica,Violència de segon ordre,Violència Vicària,Violència detectada,Detall familiar o persona propera
0,01/01/2013,2013,Entre 5 i 15 minuts,Dimarts,Gener,00/01/1900 1:00,Castellà,Barcelonès,Metropolità,Barcelona,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,01/01/2013,2013,Menys de 5 minuts,Dimarts,Gener,00/01/1900 3:20,Castellà,Barcelonès,Metropolità,Barcelona,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,01/01/2013,2013,Entre 5 i 15 minuts,Dimarts,Gener,00/01/1900 19:07,Castellà,Barcelonès,Metropolità,Barcelona,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [50]:
df_atenciones_por_anyo_drive = df_atenciones_tel_drive_1.groupby("Any").size().reset_index(name="Trucades")
df_atenciones_por_anyo_drive.head(20)

,Any,Trucades
0,2013,8597
1,2014,8741
2,2015,9741
3,2016,9310
4,2017,9104
5,2018,8716
6,2019,8789
7,2020,10606
8,2021,11109
9,2022,9867


Estos son los valores que encajan con el Flourish, para todos los años (esta es exactamente la tabla del Flourish en el momento de hacer el análisis).

In [ ]:
# TODO: son diferentes a las que hay ahora (¿cómo van a haber menos en 2023 y 2024 ahora que antes?)


In [72]:
print(len(df_atenciones_tel_drive[
    (df_atenciones_tel_drive["Any"] == 2024)
]))
print(len(df_atenciones_tel_drive[
    (df_atenciones_tel_drive["Any"] == 2022)
]))

14853
12389


Resulta que, en el momento de analizar, hay menos filas para 2024 en Dades Obertes de las que tiene IBOD guardadas (14757 vs 14853).

Hay 2 principales sospechas:

A. Se han borrado regsitros de 2023, 2024... si bien estos años no se han borrado enteros, como el periodo 2013-2021, sí se pueden haber borrado parcialmente. Está claro que esto es particularmente cierto para 2022, ya que en Dades Obertes hay 7715 filas de 2022, vs 12389 que hay en el dataset de IBOD (Drive).

B. Se han incluido múltiples veces las mismas filas, lo que significa que hay filas duplicadas. Esto es muy fácil de descartar, simplemente habría que contar las filas únicas en ambos datasets y comprobar si encaja y es lo bastante alto. Si no encaja, entonces esperamos que sea el dataset de IBOD el que tiene más filas (si fuera al revés, sería preocupante) y por ende se descarta esta opción.

In [69]:
len(df_atenciones_tel_drive[
    (df_atenciones_tel_drive["Any"] == 2024)
].drop_duplicates())

14817

Ni siquiera filtrando las filas únicas, hay menos para el dataset de IBOD que en Dades Obertes (14817 vs 14757).

### Otras vis (pendiente)

In [ ]:
query = """
SELECT
    count(*) AS total,
    `any`,
    `comarca`,
    `sexe`,
    `edat`,
    `personaquetruca`,
    `detallfamiliarpersonaquetruca`,
    `formacio`,
    `motiutrucada`,
    `relacioagressor`,
    `tempsderelacio`
GROUP BY
    `any`,
    `comarca`,
    `sexe`,
    `edat`,
    `personaquetruca`,
    `detallfamiliarpersonaquetruca`,
    `formacio`,
    `motiutrucada`,
    `relacioagressor`,
    `tempsderelacio`
ORDER BY
    `any`,
    `comarca`
LIMIT 50000
"""

response = requests.get(
    url_data,
    params={"$query": query},
    timeout=60
)
check_raw_dades_obertes(response)

df_atenciones_tel_raw = pd.DataFrame(response.json())

print(df_atenciones_tel_raw.info())
df_atenciones_tel_raw.head(20)

<class 'pandas.DataFrame'>
RangeIndex: 13279 entries, 0 to 13278
Data columns (total 11 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   total                          13279 non-null  str  
 1   any                            13279 non-null  str  
 2   comarca                        12353 non-null  str  
 3   sexe                           13279 non-null  str  
 4   edat                           13279 non-null  str  
 5   personaquetruca                13279 non-null  str  
 6   formacio                       13279 non-null  str  
 7   motiutrucada                   13250 non-null  str  
 8   relacioagressor                12460 non-null  str  
 9   tempsderelacio                 10681 non-null  str  
 10  detallfamiliarpersonaquetruca  4874 non-null   str  
dtypes: str(11)
memory usage: 2.5 MB
None


,total,any,comarca,sexe,edat,personaquetruca,formacio,motiutrucada,relacioagressor,tempsderelacio,detallfamiliarpersonaquetruca
0,1,2022,Alta Ribagorça,Dona,De 18 a 30 anys,La pròpia usuària,Estudis primaris,Violència masclista,NaN,NaN,NaN
1,1,2022,Alta Ribagorça,Dona,De 18 a 30 anys,La pròpia usuària,No consta,Violència masclista,NaN,NaN,NaN
2,1,2022,Alt Camp,Dona,Més de 60 anys,La pròpia usuària,No consta,Violència masclista,Parella,No consta,NaN
3,1,2022,Alt Camp,Dona,No consta,La pròpia usuària,Estudis superiors,No es tracta de violència,NaN,NaN,NaN
4,1,2022,Alt Camp,Dona,De 31 a 40 anys,La pròpia usuària,Estudis primaris,Violència masclista,Exparella,De 6 a 10 anys,NaN
5,2,2022,Alt Camp,Dona,De 31 a 40 anys,La pròpia usuària,Estudis secundaris,Violència masclista,Exparella,D'1 a 5 anys,NaN
6,1,2022,Alt Camp,Dona,De 31 a 40 anys,La pròpia usuària,Estudis superiors,Violència masclista,Parella,D'1 a 5 anys,NaN
7,1,2022,Alt Camp,Dona,No consta,Professional,No consta,Violència masclista,No consta,NaN,NaN
8,1,2022,Alt Camp,Dona,De 31 a 40 anys,Familiar o persona propera,No consta,Violència masclista,NaN,NaN,Amistats
9,3,2022,Alt Camp,Dona,De 31 a 40 anys,La pròpia usuària,No consta,Violència masclista,Parella,De 6 a 10 anys,NaN
